# Notebook 02 — Micro-Partitions & Natural Data Ordering
## Same Query, Two Tables — Night and Day Difference

**Key Insight**: Same data, same query — but one table was loaded in chronological order and the other was randomly shuffled. The performance difference is dramatic.

| | TRANSACTIONS (Ordered) | TRANSACTIONS_UNORDERED (Shuffled) |
|---|---|---|
| **Load pattern** | Chronological by `transaction_date` | Random (ORDER BY RANDOM()) |
| **Date-range query** | Scans ~2-3% of partitions | Scans ~100% of partitions |
| **Speed** | Seconds | 10-20x slower |

In [ ]:
USE WAREHOUSE HOL_XS;
USE SCHEMA JPMC_PERF_HOL.CONSUMER_BANKING;

---
## Step 1: Compare Clustering Quality

`SYSTEM$CLUSTERING_INFORMATION` shows how well data is physically organized for a given column.  
- **Low average depth** = data is well-ordered in partitions (great pruning)
- **High average depth** = data is scattered across partitions (poor pruning)

In [ ]:
-- ORDERED TABLE: transaction_date has excellent natural clustering
SELECT 'TRANSACTIONS (ordered)' AS table_name,
       SYSTEM$CLUSTERING_INFORMATION('TRANSACTIONS', '(transaction_date)') AS clustering_info;

In [ ]:
-- UNORDERED TABLE: transaction_date has POOR clustering (data was shuffled)
SELECT 'TRANSACTIONS_UNORDERED (shuffled)' AS table_name,
       SYSTEM$CLUSTERING_INFORMATION('TRANSACTIONS_UNORDERED', '(transaction_date)') AS clustering_info;

**What to observe**:
- Ordered table: `average_depth` close to 1-2 (dates are tightly packed in partitions)
- Unordered table: `average_depth` in the hundreds (dates are scattered everywhere)

---
## Step 2: Run the SAME Query on Both Tables

**Scenario**: "Pull all transactions from November 2024" — a standard monthly reconciliation report.

### On the ORDERED Table (Best Case)

In [ ]:
-- Disable result cache so we get true execution metrics
ALTER SESSION SET USE_CACHED_RESULT = FALSE;

In [ ]:
-- BEST CASE: Naturally ordered table — Snowflake prunes most partitions
SELECT
    transaction_type,
    channel,
    COUNT(*) AS txn_count,
    SUM(amount) AS total_amount
FROM TRANSACTIONS
WHERE transaction_date >= '2024-11-01'
  AND transaction_date < '2024-12-01'
GROUP BY 1, 2
ORDER BY total_amount DESC;

**Open the Query Profile** → Note: Partitions scanned ~2-3% of total. Fast.

### On the UNORDERED Table (Worst Case)

In [ ]:
-- WORST CASE: Same query, same data, but table was loaded in random order
SELECT
    transaction_type,
    channel,
    COUNT(*) AS txn_count,
    SUM(amount) AS total_amount
FROM TRANSACTIONS_UNORDERED
WHERE transaction_date >= '2024-11-01'
  AND transaction_date < '2024-12-01'
GROUP BY 1, 2
ORDER BY total_amount DESC;

**Open the Query Profile** → Note: Partitions scanned ~95-100% of total. Full scan!

---
## Step 3: Side-by-Side Metrics

In [ ]:
-- Compare the two queries head-to-head (using INFORMATION_SCHEMA)
-- Note: partitions_scanned/partitions_total are only in ACCOUNT_USAGE;
-- INFORMATION_SCHEMA provides bytes_scanned and timing metrics.
SELECT
    CASE
        WHEN query_text ILIKE '%TRANSACTIONS_UNORDERED%' THEN 'UNORDERED (shuffled)'
        ELSE 'ORDERED (chronological)'
    END AS table_version,
    total_elapsed_time / 1000 AS elapsed_sec,
    bytes_scanned / (1024*1024*1024) AS gb_scanned,
    compilation_time,
    execution_time
FROM TABLE(SNOWFLAKE.INFORMATION_SCHEMA.QUERY_HISTORY(
    END_TIME_RANGE_START => DATEADD(minute, -10, CURRENT_TIMESTAMP()),
    END_TIME_RANGE_END => CURRENT_TIMESTAMP(),
    RESULT_LIMIT => 50
))
WHERE query_text ILIKE '%transaction_type%channel%2024-11-01%'
  AND query_type = 'SELECT'
ORDER BY start_time DESC
LIMIT 2;

---
## Key Takeaways

| | TRANSACTIONS (Ordered) | TRANSACTIONS_UNORDERED (Shuffled) |
|---|---|---|
| **Same query** | Runs in seconds | 10-20x slower |
| **Partitions scanned** | ~2-3% | ~95-100% (full scan) |
| **Why** | Dates are packed into consecutive partitions → pruning works | Dates scattered across ALL partitions → no pruning possible |
| **Cost** | Free — just load data in the right order | Wastes compute on every single query |

**The lesson for data engineering teams**:  
How you load data determines how fast you query it. If your pipeline loads transactions in arrival order (chronological), date-range queries are essentially free. If you shuffle or randomly insert, you pay the full scan penalty on every query.

**Banking implication**:  
- Daily reconciliation, monthly statements, regulatory reporting → naturally fast on ordered table
- Same queries on a poorly-loaded table → expensive and slow, even with identical data

**Next** → [Notebook 03 — Automatic Clustering](./Notebook_03_Clustering.ipynb) — fixing the unordered case